In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D, VideoOnlyVSTProbe
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 23
test_flage=True
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))

[2026-08-17 02:02:16,058]::208875771::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [6]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/valid_set.dat","valid")
print(1)

1


In [7]:
train_set_data=getdata("/data/qx/datafrom136/data/15Framesfullandfaceandaudio/full/train_set.dat","train")
print(1)

1


In [8]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [9]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [10]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [11]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [12]:
# ============================================================
# VST-IBCA Linear Probe Configuration
# ============================================================

import copy
import os
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# Pretrained VST-IBCA checkpoint
# ------------------------------------------------------------
BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_23.pth"

# ------------------------------------------------------------
# Probe training
#
# 1e-3 -> 1e-4:
# reduce the instability observed in deeper-layer probes.
# ------------------------------------------------------------
probe_lr = 1e-4
probe_weight_decay = 1e-4
probe_epochs = 50
probe_patience = 10

probe_blocks = True
probe_stages = True

probe_save_dir = "./probe_results_vst_ibca"
os.makedirs(probe_save_dir, exist_ok=True)


# ============================================================
# 1. Load Pretrained VST-IBCA Backbone
# ============================================================

backbone = SwinTransformer3D(
    embed_dim=96,
    depths=(2, 2, 6, 2),
    num_outputs=5
).to(device)

checkpoint = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

# Compatible with:
#
# torch.save({
#     "epoch": ...,
#     "model": model.state_dict(),
#     "optimizer": ...,
#     "loss": ...
# })
if isinstance(checkpoint, dict) and "model" in checkpoint:
    backbone.load_state_dict(checkpoint["model"])
else:
    backbone.load_state_dict(checkpoint)

print("Loaded VST-IBCA checkpoint:")
print(BACKBONE_CHECKPOINT)


# ============================================================
# 2. Build Linear Probe Model
# ============================================================

model = VideoOnlyVSTProbe(
    backbone=backbone,
    embed_dim=96,
    depths=(2, 2, 6, 2),
    num_outputs=5,
    probe_blocks=probe_blocks,
    probe_stages=probe_stages
).to(device)


# ============================================================
# 3. Check Backbone is Completely Frozen
# ============================================================

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print("\n================ Probe Setup ================")
print("Backbone trainable parameters :", backbone_trainable)
print("Probe trainable parameters    :", probe_trainable)
print("Number of probe heads         :", len(model.probe_heads))
print("=============================================\n")

assert backbone_trainable == 0, \
    "ERROR: backbone is not completely frozen!"

print("Trainable parameters:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)


# ============================================================
# 4. Loss / Optimizer
# ============================================================

criterion = nn.L1Loss().to(device)

# IMPORTANT:
# Only the linear probe heads are optimized.
optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)


# ============================================================
# 5. Label Processing
# ============================================================

def prepare_labels(big_five_data, device):
    """
    Always return Big Five labels with shape [B, 5].

    Avoid squeeze(), since B=1 could otherwise become [5].
    """

    big_five_data = big_five_data.to(
        device,
        non_blocking=True
    ).float()

    batch_size = big_five_data.shape[0]

    # Compatible with [B,1,5], [B,5,1], etc.
    big_five_data = big_five_data.reshape(
        batch_size,
        -1
    )

    if big_five_data.shape[1] != 5:
        raise RuntimeError(
            f"Expected Big Five labels [B,5], "
            f"but got {tuple(big_five_data.shape)}"
        )

    return big_five_data


# ============================================================
# 6. Data Processing
# ============================================================

def prepare_batch(audiodata, fullshot, big_five_data):
    """
    Video:
        [B,T,H,W,C] -> [B,C,T,H,W]

    Audio:
        keeps its original input format for VST-IBCA.
    """

    fullshot = fullshot.permute(
        0, 4, 1, 2, 3
    ).contiguous()

    audiodata = audiodata.to(
        device,
        non_blocking=True
    )

    fullshot = fullshot.to(
        device,
        non_blocking=True
    )

    labels = prepare_labels(
        big_five_data,
        device
    )

    return audiodata, fullshot, labels


# ============================================================
# 7. Train One Probe Epoch
# ============================================================

def train_probe_epoch(model, dataloader, optimizer):
    """
    Backbone:
        completely frozen
        always eval()
        forward under no_grad()

    Probe heads:
        trainable

    Train MAE is calculated exactly like validation MAE:
        total absolute error / (N * 5)
    """

    model.train()

    absolute_errors = {
        name: 0.0
        for name in model.probe_heads.keys()
    }

    sample_count = 0

    for audiodata, fullshot, big_five_data in tqdm(
        dataloader,
        desc="Probe Train"
    ):
        audiodata, fullshot, labels = prepare_batch(
            audiodata,
            fullshot,
            big_five_data
        )

        optimizer.zero_grad(set_to_none=True)

        # ----------------------------------------------------
        # Probe predictions
        #
        # {
        #   stage1_block1: [B,5],
        #   stage1_block2: [B,5],
        #   stage1:        [B,5],
        #   ...
        # }
        # ----------------------------------------------------
        predictions = model(
            fullshot
            
        )

        probe_losses = {
            name: criterion(pred, labels)
            for name, pred in predictions.items()
        }

        # Every probe head is independent.
        # SUM keeps each probe's gradient scale unchanged.
        total_loss = torch.stack(
            list(probe_losses.values())
        ).sum()

        total_loss.backward()
        optimizer.step()

        # ----------------------------------------------------
        # Exact train MAE
        # ----------------------------------------------------
        B = labels.shape[0]
        sample_count += B

        for name, pred in predictions.items():
            absolute_errors[name] += torch.abs(
                pred.detach() - labels
            ).sum().item()

    train_results = {}

    for name, error_sum in absolute_errors.items():
        mae = error_sum / (
            sample_count * 5
        )

        train_results[name] = {
            "MAE": mae,
            "1-MAE": 1.0 - mae
        }

    return train_results


# ============================================================
# 8. Validation
# ============================================================

@torch.no_grad()
def evaluate_probes(model, dataloader):
    model.eval()

    absolute_errors = {
        name: 0.0
        for name in model.probe_heads.keys()
    }

    sample_count = 0

    for audiodata, fullshot, big_five_data in tqdm(
        dataloader,
        desc="Probe Valid"
    ):
        audiodata, fullshot, labels = prepare_batch(
            audiodata,
            fullshot,
            big_five_data
        )

        predictions = model(
            fullshot
        
        )

        B = labels.shape[0]
        sample_count += B

        for name, pred in predictions.items():
            absolute_errors[name] += torch.abs(
                pred - labels
            ).sum().item()

    results = {}

    for name, error_sum in absolute_errors.items():
        mae = error_sum / (
            sample_count * 5
        )

        results[name] = {
            "MAE": mae,
            "1-MAE": 1.0 - mae
        }

    return results


# ============================================================
# 9. Pretty Print
# ============================================================

def print_probe_results(epoch, train_results, val_results):
    print("\n")
    print("=" * 78)
    print(f"Probe Epoch {epoch}")
    print("=" * 78)

    print(
        f"{'Representation':<22}"
        f"{'Train MAE':>14}"
        f"{'Train 1-MAE':>16}"
        f"{'Val MAE':>13}"
        f"{'Val 1-MAE':>15}"
    )

    print("-" * 78)

    for name in val_results:
        print(
            f"{name:<22}"
            f"{train_results[name]['MAE']:>14.4f}"
            f"{train_results[name]['1-MAE']:>16.4f}"
            f"{val_results[name]['MAE']:>13.4f}"
            f"{val_results[name]['1-MAE']:>15.4f}"
        )

    print("=" * 78)


# ============================================================
# 10. Initialize Best Records
# ============================================================

best_mae = {
    name: float("inf")
    for name in model.probe_heads.keys()
}

best_epoch = {
    name: -1
    for name in model.probe_heads.keys()
}

best_state = {
    name: None
    for name in model.probe_heads.keys()
}

no_improve = 0
best_mean_mae = float("inf")


# ============================================================
# 11. Train Probes
# ============================================================

for epoch in range(1, probe_epochs + 1):

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------
    train_results = train_probe_epoch(
        model,
        train_dataloader,
        optimizer
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------
    val_results = evaluate_probes(
        model,
        val_dataloader
    )

    print_probe_results(
        epoch,
        train_results,
        val_results
    )

    # --------------------------------------------------------
    # Save each probe's own best state
    # --------------------------------------------------------
    for name, result in val_results.items():
        current_mae = result["MAE"]

        if current_mae < best_mae[name]:
            best_mae[name] = current_mae
            best_epoch[name] = epoch

            best_state[name] = copy.deepcopy(
                model.probe_heads[name].state_dict()
            )

    # --------------------------------------------------------
    # Global early stopping
    #
    # Only controls total training duration.
    # Each probe still uses its own best validation epoch.
    # --------------------------------------------------------
    mean_val_mae = np.mean([
        result["MAE"]
        for result in val_results.values()
    ])

    if mean_val_mae < best_mean_mae:
        best_mean_mae = mean_val_mae
        no_improve = 0
    else:
        no_improve += 1

    logger.debug(
        "VST-IBCA Probe Epoch: {} | "
        "Mean 1-MAE: {:.4f} | "
        "Patience: {}/{}".format(
            epoch,
            1.0 - mean_val_mae,
            no_improve,
            probe_patience
        )
    )

    if no_improve >= probe_patience:
        print(
            f"\nProbe early stopping at epoch {epoch}."
        )
        break


# ============================================================
# 12. Restore Each Probe's Own Best State
# ============================================================

for name in model.probe_heads:
    if best_state[name] is not None:
        model.probe_heads[name].load_state_dict(
            best_state[name]
        )


# ============================================================
# 13. Final Evaluation
# ============================================================

final_results = evaluate_probes(
    model,
    val_dataloader
)


print("\n\n")
print("=" * 76)
print("FINAL VST-IBCA LINEAR PROBE RESULTS")
print("=" * 76)

print(
    f"{'Representation':<22}"
    f"{'Best Epoch':>12}"
    f"{'MAE':>14}"
    f"{'1-MAE':>14}"
)

print("-" * 76)

for name, result in final_results.items():
    print(
        f"{name:<22}"
        f"{best_epoch[name]:>12}"
        f"{result['MAE']:>14.4f}"
        f"{result['1-MAE']:>14.4f}"
    )

print("=" * 76)


# ============================================================
# 14. Save Probe Heads + Results
# ============================================================

save_path = os.path.join(
    probe_save_dir,
    "vst_ibca_linear_probes.pth"
)

torch.save(
    {
        # Only the probe heads are saved.
        "probe_heads": model.probe_heads.state_dict(),

        "best_mae": best_mae,
        "best_epoch": best_epoch,
        "final_results": final_results,

        # Pretrained frozen backbone used for probing
        "backbone_checkpoint": BACKBONE_CHECKPOINT,

        # Probe configuration
        "probe_lr": probe_lr,
        "probe_weight_decay": probe_weight_decay,
        "probe_epochs": probe_epochs,

        "probe_blocks": probe_blocks,
        "probe_stages": probe_stages
    },
    save_path
)

print("\nSaved VST-IBCA probe model:")
print(save_path)

Loaded VST-IBCA checkpoint:
./save_swintransformer_folder/model_23.pth

================ Probe Setup ================
Backbone trainable parameters : 0
Probe trainable parameters    : 36560
Number of probe heads         : 16

Trainable parameters:
probe_heads.stage1_block1.weight
probe_heads.stage1_block1.bias
probe_heads.stage1_block2.weight
probe_heads.stage1_block2.bias
probe_heads.stage1.weight
probe_heads.stage1.bias
probe_heads.stage2_block1.weight
probe_heads.stage2_block1.bias
probe_heads.stage2_block2.weight
probe_heads.stage2_block2.bias
probe_heads.stage2.weight
probe_heads.stage2.bias
probe_heads.stage3_block1.weight
probe_heads.stage3_block1.bias
probe_heads.stage3_block2.weight
probe_heads.stage3_block2.bias
probe_heads.stage3_block3.weight
probe_heads.stage3_block3.bias
probe_heads.stage3_block4.weight
probe_heads.stage3_block4.bias
probe_heads.stage3_block5.weight
probe_heads.stage3_block5.bias
probe_heads.stage3_block6.weight
probe_heads.stage3_block6.bias
probe_heads.

Probe Valid: 100%|██████████| 498/498 [00:46<00:00, 10.75it/s]
[2026-08-17 02:05:17,636]::2861535557::DEBUG::VST-IBCA Probe Epoch: 1 | Mean 1-MAE: 0.8629 | Patience: 0/10




Probe Epoch 1
Representation             Train MAE     Train 1-MAE      Val MAE      Val 1-MAE
------------------------------------------------------------------------------
stage1_block1                 0.4194          0.5806       0.2229         0.7771
stage1_block2                 0.4003          0.5997       0.2070         0.7930
stage1                        0.1464          0.8536       0.1196         0.8804
stage2_block1                 0.1300          0.8700       0.1147         0.8853
stage2_block2                 0.1433          0.8567       0.1175         0.8825
stage2                        0.1805          0.8195       0.1514         0.8486
stage3_block1                 0.1557          0.8443       0.1321         0.8679
stage3_block2                 0.1755          0.8245       0.1456         0.8544
stage3_block3                 0.1848          0.8152       0.1497         0.8503
stage3_block4                 0.1849          0.8151       0.1465         0.8535
stage3_block5 

Probe Valid: 100%|██████████| 498/498 [00:46<00:00, 10.76it/s]
[2026-08-17 02:08:06,515]::2861535557::DEBUG::VST-IBCA Probe Epoch: 2 | Mean 1-MAE: 0.8705 | Patience: 0/10




Probe Epoch 2
Representation             Train MAE     Train 1-MAE      Val MAE      Val 1-MAE
------------------------------------------------------------------------------
stage1_block1                 0.1947          0.8053       0.1726         0.8274
stage1_block2                 0.1809          0.8191       0.1642         0.8358
stage1                        0.0927          0.9073       0.1121         0.8879
stage2_block1                 0.0861          0.9139       0.1057         0.8943
stage2_block2                 0.0876          0.9124       0.1090         0.8910
stage2                        0.1109          0.8891       0.1413         0.8587
stage3_block1                 0.1060          0.8940       0.1266         0.8734
stage3_block2                 0.1181          0.8819       0.1279         0.8721
stage3_block3                 0.1219          0.8781       0.1479         0.8521
stage3_block4                 0.1181          0.8819       0.1447         0.8553
stage3_block5 

...



Probe Epoch 49
Representation             Train MAE     Train 1-MAE      Val MAE      Val 1-MAE
------------------------------------------------------------------------------
stage1_block1                 0.0845          0.9155       0.1015         0.8985
stage1_block2                 0.0837          0.9163       0.0985         0.9015
stage1                        0.0580          0.9420       0.0901         0.9099
stage2_block1                 0.0541          0.9459       0.0895         0.9105
stage2_block2                 0.0542          0.9458       0.0900         0.9100
stage2                        0.0557          0.9443       0.0994         0.9006
stage3_block1                 0.0641          0.9359       0.0932         0.9068
stage3_block2                 0.0663          0.9337       0.0977         0.9023
stage3_block3                 0.0667          0.9333       0.0931         0.9069
stage3_block4                 0.0672          0.9328       0.0983         0.9017
stage3_block5

Probe Valid: 100%|██████████| 498/498 [00:46<00:00, 10.73it/s]
[2026-08-17 04:27:13,705]::2861535557::DEBUG::VST-IBCA Probe Epoch: 50 | Mean 1-MAE: 0.9011 | Patience: 4/10




Probe Epoch 50
Representation             Train MAE     Train 1-MAE      Val MAE      Val 1-MAE
------------------------------------------------------------------------------
stage1_block1                 0.0841          0.9159       0.1001         0.8999
stage1_block2                 0.0832          0.9168       0.0985         0.9015
stage1                        0.0578          0.9422       0.0909         0.9091
stage2_block1                 0.0539          0.9461       0.0898         0.9102
stage2_block2                 0.0541          0.9459       0.0898         0.9102
stage2                        0.0557          0.9443       0.0925         0.9075
stage3_block1                 0.0624          0.9376       0.1028         0.8972
stage3_block2                 0.0658          0.9342       0.0941         0.9059
stage3_block3                 0.0664          0.9336       0.1083         0.8917
stage3_block4                 0.0669          0.9331       0.1004         0.8996
stage3_block5

Probe Valid: 100%|██████████| 498/498 [00:46<00:00, 10.77it/s]




FINAL VST-IBCA LINEAR PROBE RESULTS
Representation          Best Epoch           MAE         1-MAE
----------------------------------------------------------------------------
stage1_block1                   43        0.0990        0.9010
stage1_block2                   49        0.0985        0.9015
stage1                          49        0.0901        0.9099
stage2_block1                   49        0.0895        0.9105
stage2_block2                   50        0.0898        0.9102
stage2                          38        0.0922        0.9078
stage3_block1                   49        0.0932        0.9068
stage3_block2                   50        0.0941        0.9059
stage3_block3                   49        0.0931        0.9069
stage3_block4                   46        0.0957        0.9043
stage3_block5                   32        0.0958        0.9042
stage3_block6                   36        0.0965        0.9035
stage3                           8        0.0920        0.9080
st